In [ ]:
from utils import ByteSeq
import pickle

# Shared BPE (single vocab trained on en+yue concatenated, produced by run_bpe.py)
with open("shared.pkl", "rb") as f:
    shared = pickle.load(f)

en_chars = yue_chars = shared["chars"]
en_tokens = yue_tokens = shared["tokens"]

In [ ]:
import torch.nn.functional as F
from torch import nn
import numpy as np
import torch

class Embedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        # paper-style init: small weights so tied output_proj logits stay sane
        self.weight = nn.Parameter(torch.randn(vocab_size, d_model) * (d_model ** -0.5))
        self.d_model = d_model

    def forward(self, x):
        return self.weight[x] * (self.d_model ** 0.5)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len):
        super().__init__()
        pe = torch.zeros(max_seq_len, d_model)
        for pos in range(max_seq_len):
            for k in range(d_model // 2):
                freq = pos / (10000 ** (2*k / d_model))
                pe[pos, 2*k] = np.sin(freq)
                pe[pos, 2*k+1] = np.cos(freq)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, seq_len, n_heads, dropout=0.1, masked=False):
        super().__init__()
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model) 
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.out_dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool())
        self.masked = masked

    def forward(self, x, cross=None):
        B, S, D = x.shape
        cross = cross if cross is not None else x
        Q = self.W_Q(x)
        K = self.W_K(cross)
        V = self.W_V(cross)
        
        # want B, n_heads, S, d_k
        Q = Q.view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(B, cross.size(1), self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(B, cross.size(1), self.n_heads, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)
        if self.masked:
            scores = scores.masked_fill(self.mask[:S, :S], float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ V
        # back to B, S, D
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return self.out_dropout(self.W_O(out))

class Encoder(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.1, d_ff=1024):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.source_MHA = MultiHeadAttention(d_model, max_seq_len, n_heads, dropout=dropout, masked=False)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.ff_dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.source_MHA(self.norm1(x))
        x = x + self.ff_dropout(self.ff(self.norm2(x)))
        return x

class Decoder(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.1, d_ff=1024):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.masked_MHA = MultiHeadAttention(d_model, max_seq_len, n_heads, dropout=dropout, masked=True)
        self.cross_MHA = MultiHeadAttention(d_model, max_seq_len, n_heads, dropout=dropout, masked=False)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.ff_dropout = nn.Dropout(dropout)

    def forward(self, target, cross):
        target = target + self.masked_MHA(self.norm1(target))
        target = target + self.cross_MHA(self.norm2(target), cross)
        target = target + self.ff_dropout(self.ff(self.norm3(target)))
        return target


class Transformer(nn.Module):
    def __init__(self, source_vocab_size, target_vocab_size, d_model, n_heads, source_l_layers, target_l_layers, max_seq_len, dropout=0.1):
        super().__init__()
        self.target_embed = Embedding(target_vocab_size, d_model)
        # when vocab is shared, reuse the same embedding tensor for source too
        self.source_embed = self.target_embed if source_vocab_size == target_vocab_size else Embedding(source_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_seq_len)
        self.embed_dropout = nn.Dropout(dropout)
        self.encoder_layers = nn.ModuleList([Encoder(d_model, n_heads, max_seq_len, dropout=dropout) for _ in range(source_l_layers)])
        self.decoder_layers = nn.ModuleList([Decoder(d_model, n_heads, max_seq_len, dropout=dropout) for _ in range(target_l_layers)])
        # Tie output projection to target embedding (Press & Wolf 2017, Inan et al. 2017)
        self.output_proj = nn.Linear(d_model, target_vocab_size, bias=False)
        self.output_proj.weight = self.target_embed.weight
    
    def encoder(self, source):
        source = self.embed_dropout(self.pos_enc(self.source_embed(source)))
        for layer in self.encoder_layers:
            source = layer(source)
        return source

    def decoder(self, target, enc_out):
        target = self.embed_dropout(self.pos_enc(self.target_embed(target)))
        for layer in self.decoder_layers:
            target = layer(target, enc_out)
        return self.output_proj(target)

    def forward(self, source, target):
        enc_out = self.encoder(source)
        dec_out = self.decoder(target, enc_out)
        return dec_out

In [ ]:
from utils import token_split

PAD_TOKEN = "<PAD>"
START_EN_TOKEN = "<2EN>"
START_YUE_TOKEN = "<2YUE>"
END_TOKEN = "<END>"
PAD_ID = 0

vocab_list = [PAD_TOKEN] + en_tokens + sorted(en_chars) + [START_EN_TOKEN, START_YUE_TOKEN, END_TOKEN]
vocab = {token: i for i, token in enumerate(vocab_list)}
merges = en_tokens


def encode(text, vocab, merges):
    merge_priority = {merge: i for i, merge in enumerate(merges)}
    words = token_split(text)
    all_ids = []
    for word in words:
        tokens = [ByteSeq(b) for b in word.encode("utf-8")]
        while len(tokens) > 1:
            best = None
            best_idx = None
            for i in range(len(tokens) - 1):
                pair = tokens[i] + tokens[i+1]
                if pair in merge_priority:
                    if best is None or merge_priority[pair] < merge_priority[best]:
                        best = pair
                        best_idx = i
            if best is None:
                break
            tokens[best_idx] = best
            tokens.pop(best_idx + 1)
        for t in tokens:
            all_ids.append(vocab[t])
    return all_ids

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

class TranslationDataset(Dataset):
    def __init__(self, en_file, yue_file, vocab, merges):
        with open(en_file, encoding="utf-8") as f:
            en_lines = f.read().strip().split("\n")
        with open(yue_file, encoding="utf-8") as f:
            yue_lines = f.read().strip().split("\n")

        self.data = []
        for en, yue in tqdm(zip(en_lines, yue_lines), total = len(en_lines)):
            en_src = torch.tensor(encode(en, vocab, merges))
            yue_tgt = torch.tensor(
                [vocab[START_YUE_TOKEN]] +
                encode(yue, vocab, merges) +
                [vocab[END_TOKEN]]
            )
            self.data.append((en_src, yue_tgt))

            yue_src = torch.tensor(encode(yue, vocab, merges))
            en_tgt = torch.tensor(
                [vocab[START_EN_TOKEN]] +
                encode(en, vocab, merges) +
                [vocab[END_TOKEN]]
            )

            self.data.append((yue_src, en_tgt))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    srcs, tgts = zip(*batch)
    srcs = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD_ID)
    tgts = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=PAD_ID)
    return srcs, tgts


dataset = TranslationDataset("en.txt", "yue.txt", vocab, merges)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_set, batch_size=128, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=128, collate_fn=collate_fn)

In [ ]:
D_MODEL = 256
N_HEADS = 8
N_LAYERS = 3
DROPOUT = 0.2

model = Transformer(
    source_vocab_size=len(vocab),
    target_vocab_size=len(vocab),
    d_model=D_MODEL,
    n_heads=N_HEADS,
    source_l_layers=N_LAYERS,
    target_l_layers=N_LAYERS,
    max_seq_len=512,
    dropout=DROPOUT
)

device = torch.device("cuda")
model = model.to(device)

# warmup schedule from "Attention Is All You Need" — formula IS the lr (use base lr=1.0)
WARMUP_STEPS = 4000
optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)

def lr_lambda(step):
    step = max(step, 1)
    return D_MODEL ** (-0.5) * min(step ** (-0.5), step * WARMUP_STEPS ** (-1.5))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)

EPOCHS = 25
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for src, tgt in tqdm(train_loader):
        src, tgt = src.to(device), tgt.to(device)
        tgt_input = tgt[:, :-1]
        tgt_label = tgt[:, 1:]

        logits = model(src, tgt_input)
        loss = loss_fn(logits.reshape(-1, len(vocab)), tgt_label.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)
            tgt_input = tgt[:, :-1] # mask out last token for decoder input
            tgt_label = tgt[:, 1:] # don't need start token for label
            logits = model(src, tgt_input)
            val_loss += loss_fn(logits.reshape(-1, len(vocab)), tgt_label.reshape(-1)).item()

    print(f"Epoch {epoch+1} | Train loss: {total_loss/len(train_loader):.4f} | Val loss: {val_loss/len(val_loader):.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")